In [1]:
import os

import pandas as pd
import psycopg
from dotenv import load_dotenv

load_dotenv()

# mesma conexão do extrator (se o notebook da NeoWs já tem uma, pode reaproveitar)
conn = psycopg.connect(
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=os.getenv("POSTGRES_PORT", "5436"),
    dbname=os.getenv("POSTGRES_DB"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
)

# a bronze guarda um snapshot por execução; pegamos o mais recente da consulta cad_terra
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT id, query_params, payload, ingested_at
        FROM bronze.cad_raw
        WHERE source = 'cad_terra'
        ORDER BY id DESC
        LIMIT 1
        """
    )
    linha_id, params, payload, ingested_at = cur.fetchone()

print("linha:", linha_id, "| gravada em:", ingested_at)
print("parâmetros:", params)
print("assinatura:", payload["signature"])
print("count:", payload["count"])

linha: 1 | gravada em: 2026-11-17 03:34:17.342628+00:00
parâmetros: {'neo': 'true', 'body': 'Earth', 'sort': 'date', 'date-max': '2027-01-16', 'date-min': '2026-11-17', 'diameter': 'true', 'dist-max': '0.05', 'fullname': 'true'}
assinatura: {'source': 'NASA/JPL SBDB Close Approach Data API', 'version': '1.5'}
count: 33


In [2]:
# "fields" são os nomes das colunas e "data" é a lista de linhas
cad_df = pd.DataFrame(payload["data"], columns=payload["fields"])

print(cad_df.shape)
cad_df.head()

(33, 14)


,des,orbit_id,jd,cd,dist,dist_min,dist_max,v_rel,v_inf,t_sigma_f,h,diameter,diameter_sigma,fullname
0,2026 VJ13,5,2461361.733296706,2026-Nov-17 05:36,0.0090048004764615,0.00897806575675958,0.00903153433681345,10.4341227005691,10.405725619296,< 00:01,27.242,None,None,(2026 VJ13)
1,2023 VE6,4,2461361.781963406,2026-Nov-17 06:46,0.0470991947664407,0.0153912148117894,0.0788139034845923,10.1012078051812,10.0956057676574,5_03:28,27.16,None,None,(2023 VE6)
2,2026 WA,1,2461361.832909882,2026-Nov-17 07:59,0.000844938687608272,0.000843123647235508,0.00084675372276953,6.70807169244967,6.22023377657568,00:02,29.151,None,None,(2026 WA)
3,2026 VB13,3,2461361.989943428,2026-Nov-17 11:46,0.0254136235770872,0.0251944378489589,0.02563280742127,20.4726551486594,20.4675333096672,< 00:01,25.262,None,None,(2026 VB13)
4,2022 VB2,17,2461362.148595456,2026-Nov-17 15:34,0.0198705192578925,0.0198704620476564,0.0198705764681489,8.95050373953794,8.93550966780381,< 00:01,25.17,None,None,(2022 VB2)


In [3]:
cad_df.dtypes

des                  str
orbit_id             str
jd                   str
cd                   str
dist                 str
dist_min             str
dist_max             str
v_rel                str
v_inf                str
t_sigma_f            str
h                    str
diameter          object
diameter_sigma    object
fullname             str
dtype: object

In [4]:
cad = cad_df.copy()  # copia para não mexer no DataFrame original

# identificador da órbita: inteiro
cad["orbit_id"] = cad["orbit_id"].astype("int64")

# data juliana: número decimal
cad["jd"] = pd.to_numeric(cad["jd"])

# data/hora do encontro: de "2026-Nov-17 05:36" para datetime
cad["cd"] = pd.to_datetime(cad["cd"], format="%Y-%b-%d %H:%M")

# colunas numéricas; o None vira NaN (valor ausente do pandas)
colunas_numericas = [
    "dist",
    "dist_min",
    "dist_max",  # distâncias em au
    "v_rel",
    "v_inf",  # velocidades em km/s
    "h",  # magnitude absoluta
    "diameter",
    "diameter_sigma",  # em km, None quando desconhecido
]
cad[colunas_numericas] = cad[colunas_numericas].apply(pd.to_numeric)

# a API põe espaços à esquerda no nome completo (alinhamento em tabela)
cad["fullname"] = cad["fullname"].str.strip()

cad.dtypes

des                          str
orbit_id                   int64
jd                       float64
cd                datetime64[us]
dist                     float64
dist_min                 float64
dist_max                 float64
v_rel                    float64
v_inf                    float64
t_sigma_f                    str
h                        float64
diameter                 float64
diameter_sigma           float64
fullname                     str
dtype: object

In [5]:
cad.isna().sum()

des                0
orbit_id           0
jd                 0
cd                 0
dist               0
dist_min           0
dist_max           0
v_rel              0
v_inf              0
t_sigma_f          0
h                  0
diameter          33
diameter_sigma    33
fullname           0
dtype: int64

In [6]:
# o mesmo objeto pode passar mais de uma vez na janela (des repete),
# mas o mesmo objeto no mesmo instante (des + jd) não deveria repetir
print("des repetidos:", cad["des"].duplicated().sum())
print("des + jd repetidos:", cad.duplicated(subset=["des", "jd"]).sum())

des repetidos: 0
des + jd repetidos: 0


In [7]:
# a distância nominal deve estar entre o mínimo e o máximo (3 sigma)
coerente = (cad["dist_min"] <= cad["dist"]) & (cad["dist"] <= cad["dist_max"])
print("linhas incoerentes:", (~coerente).sum())

# o filtro dist-max=0.05 vale para a distância NOMINAL (dist), não para dist_max
print("maior dist:", cad["dist"].max(), "(limite pedido: 0.05)")
print("dist <= 0.05 em todas:", (cad["dist"] <= 0.05).all())

linhas incoerentes: 0
maior dist: 0.0478012298502166 (limite pedido: 0.05)
dist <= 0.05 em todas: True


In [8]:
print("pedido:", params["date-min"], "->", params["date-max"])
print("recebido:", cad["cd"].min(), "->", cad["cd"].max())
print("ordenado por data:", cad["cd"].is_monotonic_increasing)

pedido: 2026-11-17 -> 2027-01-16
recebido: 2026-11-17 05:36:00 -> 2027-01-13 19:24:00
ordenado por data: True


In [9]:
cad[["dist", "v_rel", "v_inf", "h"]].describe()

,dist,v_rel,v_inf,h
count,33.000000,33.000000,33.000000,33.000000
mean,0.026887,9.995966,9.949878,25.839879
std,0.013116,4.023718,4.085080,2.290987
min,0.000845,0.796700,0.212231,19.860000
25%,0.017743,7.671573,7.662826,24.727000
50%,0.026110,10.203431,10.170547,25.539000
75%,0.037148,11.672661,11.655627,27.160000
max,0.047801,20.472655,20.467533,31.367000
